# GuardPlant — Plant Species Classification Benchmark

## Step 1: Environment setup and data pipeline

This notebook benchmarks two image-classification systems for the GuardPlant mobile application:

1. **Ultralytics YOLOv8 Classifier** (`yolov8n-cls`)
2. **Torchvision EfficientNet-B0**

Both models will learn the same **200 plant species** represented in the GuardPlant botanical database (`plants_en.json`). The source archive is expected at `/content/drive/MyDrive/dataset.zip` and must contain a pre-split ImageFolder-style dataset:

```text
dataset/
├── train/<plant_id>/...   # 80%
├── val/<plant_id>/...     # 10%
└── test/<plant_id>/...    # 10%
```

The class folders are treated as canonical plant IDs. Their alphabetically sorted order is persisted to `class_names.json`, ensuring that training, evaluation, ONNX export, and mobile inference all use the same numeric label mapping. Step 1 mounts Google Drive, extracts and validates the dataset, then creates reproducible PyTorch datasets and data loaders for EfficientNet-B0. The same validated dataset root will be passed to Ultralytics in a later step.


## 1. Install and import dependencies

Google Colab already includes PyTorch. Ultralytics and ONNX are installed now so later benchmark steps use the same runtime. If Colab asks for a runtime restart after installation, restart it and continue from this cell.


In [ ]:
%pip install -q "ultralytics>=8.2,<9" "onnx>=1.16,<2" "tabulate>=0.9,<1"


In [ ]:
import json
import os
import random
import zipfile
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
BATCH_SIZE = 32
IMAGE_SIZE = 224
EXPECTED_NUM_CLASSES = 200

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. Mount Google Drive and extract the dataset

Extraction is skipped on reruns when all three split folders already exist. Set `FORCE_REEXTRACT = True` to refresh files from the ZIP; existing files with the same paths will be overwritten, while unrelated files are preserved.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

ZIP_PATH = Path("/content/drive/MyDrive/dataset.zip")
EXTRACT_DIR = Path("/content/dataset")
FORCE_REEXTRACT = False
REQUIRED_SPLITS = ("train", "val", "test")

if not ZIP_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset archive not found: {ZIP_PATH}. "
        "Upload dataset.zip to the root of Google Drive and rerun this cell."
    )

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
possible_dataset_roots = (EXTRACT_DIR, EXTRACT_DIR / "dataset")
splits_already_present = any(
    all((root / split).is_dir() for split in REQUIRED_SPLITS)
    for root in possible_dataset_roots
)

if FORCE_REEXTRACT or not splits_already_present:
    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        bad_member = archive.testzip()
        if bad_member is not None:
            raise zipfile.BadZipFile(f"CRC check failed for archive member: {bad_member}")
        archive.extractall(EXTRACT_DIR)
    print(f"Extracted {ZIP_PATH} into {EXTRACT_DIR}")
else:
    print(f"Dataset splits already exist in {EXTRACT_DIR}; extraction skipped.")

# Support either train/val/test at the archive root or inside one dataset/ folder.
DATASET_ROOT = next(
    (root for root in possible_dataset_roots if all((root / split).is_dir() for split in REQUIRED_SPLITS)),
    None,
)
if DATASET_ROOT is None:
    raise FileNotFoundError(
        f"Could not find train/val/test under {EXTRACT_DIR}. "
        "Check the directory layout inside dataset.zip."
    )

print(f"Validated dataset root: {DATASET_ROOT}")


## 3. Validate splits and lock the class mapping

This validation rejects missing/extra class directories, empty classes, and any class count other than 200. Image counts include the standard image formats supported by the pipeline. `class_names.json` stores a JSON array where each position is the integer class index and each value is the corresponding plant ID.


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp"}


def class_directories(split_dir: Path) -> list[Path]:
    return sorted((path for path in split_dir.iterdir() if path.is_dir()), key=lambda path: path.name)


def count_images(directory: Path) -> int:
    return sum(
        1
        for path in directory.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


split_class_names = {}
split_image_counts = {}
empty_classes = {}

for split in REQUIRED_SPLITS:
    split_dir = DATASET_ROOT / split
    classes = class_directories(split_dir)
    split_class_names[split] = [path.name for path in classes]
    per_class_counts = {path.name: count_images(path) for path in classes}
    split_image_counts[split] = sum(per_class_counts.values())
    empty_classes[split] = [name for name, count in per_class_counts.items() if count == 0]

class_names = split_class_names["train"]
if len(class_names) != EXPECTED_NUM_CLASSES:
    raise ValueError(
        f"Expected {EXPECTED_NUM_CLASSES} train classes, found {len(class_names)}."
    )

train_class_set = set(class_names)
for split in REQUIRED_SPLITS:
    split_class_set = set(split_class_names[split])
    missing = sorted(train_class_set - split_class_set)
    extra = sorted(split_class_set - train_class_set)
    if missing or extra:
        raise ValueError(
            f"Class mismatch in '{split}': missing={missing}, extra={extra}"
        )
    if empty_classes[split]:
        raise ValueError(f"Empty class folders in '{split}': {empty_classes[split]}")

CLASS_NAMES_PATH = Path.cwd() / "class_names.json"
with CLASS_NAMES_PATH.open("w", encoding="utf-8") as file:
    json.dump(class_names, file, ensure_ascii=False, indent=2)

print(f"Dataset root: {DATASET_ROOT}")
for split in REQUIRED_SPLITS:
    print(
        f"{split:>5}: {split_image_counts[split]:,} images "
        f"across {len(split_class_names[split])} classes"
    )
print(f"Saved canonical class-index mapping to: {CLASS_NAMES_PATH}")
print("Mapping preview:", {index: name for index, name in enumerate(class_names[:5])})


## 4. Prepare EfficientNet-B0 datasets and data loaders

Training images receive light geometric augmentation. Validation and test images use deterministic preprocessing. All splits are resized to 224×224 and normalized with ImageNet statistics, matching EfficientNet-B0's pretrained weights.


In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

train_dataset = datasets.ImageFolder(DATASET_ROOT / "train", transform=train_transform)
val_dataset = datasets.ImageFolder(DATASET_ROOT / "val", transform=eval_transform)
test_dataset = datasets.ImageFolder(DATASET_ROOT / "test", transform=eval_transform)

expected_class_to_idx = {name: index for index, name in enumerate(class_names)}
for split, dataset in (
    ("train", train_dataset),
    ("val", val_dataset),
    ("test", test_dataset),
):
    if dataset.class_to_idx != expected_class_to_idx:
        raise RuntimeError(f"ImageFolder label mapping differs from class_names.json for {split}.")

NUM_WORKERS = min(2, os.cpu_count() or 1)
loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": DEVICE.type == "cuda",
    "persistent_workers": NUM_WORKERS > 0,
}

data_loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    generator=data_loader_generator,
    **loader_kwargs,
)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

YOLO_DATASET_ROOT = str(DATASET_ROOT)

print(f"Classes: {len(class_names)}")
print(f"DataLoader workers: {NUM_WORKERS}")
print(f"Batches — train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")
print(f"Ultralytics classification dataset root: {YOLO_DATASET_ROOT}")


## 5. Smoke-test one batch

Loading one batch catches early pipeline problems and confirms the tensor and label shapes before a training run begins.


In [ ]:
sample_images, sample_labels = next(iter(train_loader))

assert sample_images.ndim == 4 and sample_images.shape[1:] == (3, IMAGE_SIZE, IMAGE_SIZE)
assert sample_labels.min().item() >= 0
assert sample_labels.max().item() < EXPECTED_NUM_CLASSES

print(f"Image batch shape: {tuple(sample_images.shape)}")
print(f"Label batch shape: {tuple(sample_labels.shape)}")
print("First labels:", [class_names[index] for index in sample_labels[:5].tolist()])
print("Step 1 complete: the dataset and EfficientNet data pipeline are ready.")


# Step 2: Model training

This step trains both benchmark models for 20 epochs on the validated splits from Step 1. Ultralytics manages the YOLOv8 classification loop and writes its native logs, plots, and checkpoints under `guardplant_artifacts/yolo/`. The native PyTorch loop records loss, Top-1 accuracy, and Top-5 accuracy for EfficientNet-B0.

The requested portable checkpoint names are written in the current project directory:

- `yolov8_best.pt` — copy of Ultralytics' best validation checkpoint
- `efficientnet_best.pth` — best EfficientNet checkpoint selected by validation Top-1 accuracy

> **Colab recommendation:** Use a GPU runtime and run every Step 1 cell before starting Step 2.


In [ ]:
import shutil

from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0
from tqdm.auto import tqdm
from ultralytics import YOLO

TRAIN_EPOCHS = 20
ARTIFACT_ROOT = Path.cwd() / "guardplant_artifacts"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

YOLO_BEST_PATH = Path.cwd() / "yolov8_best.pt"
EFFICIENTNET_BEST_PATH = Path.cwd() / "efficientnet_best.pth"
EFFICIENTNET_HISTORY_PATH = ARTIFACT_ROOT / "efficientnet_history.json"

print(f"Training device: {DEVICE}")
print(f"Artifact directory: {ARTIFACT_ROOT}")


## Model A — Ultralytics YOLOv8 Nano Classifier

Ultralytics consumes the `train`, `val`, and `test` folders beneath `YOLO_DATASET_ROOT`. Its run directory retains `results.csv`, generated plots, configuration, and both `best.pt` and `last.pt`. After training, the best checkpoint is copied to the required `yolov8_best.pt` path and its class order is verified against `class_names.json`.


In [ ]:
yolo_model = YOLO("yolov8n-cls.pt")

yolo_train_results = yolo_model.train(
    data=YOLO_DATASET_ROOT,
    epochs=TRAIN_EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=NUM_WORKERS,
    device=0 if DEVICE.type == "cuda" else "cpu",
    project=str(ARTIFACT_ROOT / "yolo"),
    name="yolov8n_cls_20epochs",
    exist_ok=True,
    seed=SEED,
    deterministic=True,
    save=True,
    plots=True,
)

YOLO_RUN_DIR = Path(yolo_model.trainer.save_dir)
ultralytics_best_path = YOLO_RUN_DIR / "weights" / "best.pt"
if not ultralytics_best_path.is_file():
    raise FileNotFoundError(f"Ultralytics best checkpoint was not created: {ultralytics_best_path}")

shutil.copy2(ultralytics_best_path, YOLO_BEST_PATH)

yolo_best_model = YOLO(str(YOLO_BEST_PATH))
yolo_class_names = [yolo_best_model.names[index] for index in range(len(yolo_best_model.names))]
if yolo_class_names != class_names:
    raise RuntimeError("YOLO checkpoint class order does not match class_names.json.")

print(f"YOLO run artifacts: {YOLO_RUN_DIR}")
print(f"YOLO best checkpoint: {YOLO_BEST_PATH}")
print(f"Verified YOLO class mapping for {len(yolo_class_names)} classes.")

# Release YOLO's GPU allocations before initializing EfficientNet.
del yolo_model, yolo_best_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Model B — Torchvision EfficientNet-B0

The ImageNet-pretrained backbone is retained and the final linear layer is replaced with a 200-class head. Training uses cross-entropy loss, AdamW with a learning rate of `1e-3`, and cosine annealing over the full 20-epoch run.


In [ ]:
efficientnet_weights = EfficientNet_B0_Weights.DEFAULT
efficientnet_model = efficientnet_b0(weights=efficientnet_weights)
classifier_in_features = efficientnet_model.classifier[1].in_features
efficientnet_model.classifier[1] = nn.Linear(classifier_in_features, EXPECTED_NUM_CLASSES)
efficientnet_model = efficientnet_model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(efficientnet_model.parameters(), lr=1e-3)
scheduler = CosineAnnealingLR(optimizer, T_max=TRAIN_EPOCHS)

print(f"EfficientNet output classes: {efficientnet_model.classifier[1].out_features}")
print(f"Trainable parameters: {sum(p.numel() for p in efficientnet_model.parameters() if p.requires_grad):,}")


### Native PyTorch training and validation loop

Each pass computes sample-weighted mean loss and exact Top-1/Top-5 accuracy. The progress bars show running metrics, while the compact epoch summary is retained in notebook output.


In [ ]:
def run_classification_epoch(model, data_loader, loss_function, device, optimizer=None, phase="train"):
    is_training = optimizer is not None
    model.train(mode=is_training)

    running_loss = 0.0
    top1_correct = 0
    top5_correct = 0
    sample_count = 0

    progress = tqdm(data_loader, desc=phase, leave=False)
    for images, targets in progress:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            logits = model(images)
            loss = loss_function(logits, targets)
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite {phase} loss encountered: {loss.item()}")
            if is_training:
                loss.backward()
                optimizer.step()

        batch_size = targets.size(0)
        sample_count += batch_size
        running_loss += loss.item() * batch_size

        top5_predictions = logits.topk(k=5, dim=1, largest=True, sorted=True).indices
        matches = top5_predictions.eq(targets.unsqueeze(1))
        top1_correct += matches[:, :1].any(dim=1).sum().item()
        top5_correct += matches.any(dim=1).sum().item()

        progress.set_postfix(
            loss=f"{running_loss / sample_count:.4f}",
            top1=f"{100.0 * top1_correct / sample_count:.2f}%",
            top5=f"{100.0 * top5_correct / sample_count:.2f}%",
        )

    if sample_count == 0:
        raise RuntimeError(f"The {phase} data loader produced no samples.")

    return {
        "loss": running_loss / sample_count,
        "top1": 100.0 * top1_correct / sample_count,
        "top5": 100.0 * top5_correct / sample_count,
    }


In [ ]:
efficientnet_history = {
    "train_loss": [],
    "val_loss": [],
    "train_top1": [],
    "val_top1": [],
    "train_top5": [],
    "val_top5": [],
    "learning_rate": [],
}
best_val_top1 = float("-inf")
best_epoch = 0

for epoch in range(1, TRAIN_EPOCHS + 1):
    current_lr = optimizer.param_groups[0]["lr"]
    print(f"\nEpoch {epoch:02d}/{TRAIN_EPOCHS} | lr={current_lr:.6f}")

    train_metrics = run_classification_epoch(
        efficientnet_model, train_loader, criterion, DEVICE, optimizer=optimizer, phase="train"
    )
    val_metrics = run_classification_epoch(
        efficientnet_model, val_loader, criterion, DEVICE, optimizer=None, phase="val"
    )

    for metric_name in ("loss", "top1", "top5"):
        efficientnet_history[f"train_{metric_name}"].append(train_metrics[metric_name])
        efficientnet_history[f"val_{metric_name}"].append(val_metrics[metric_name])
    efficientnet_history["learning_rate"].append(current_lr)

    scheduler.step()

    if val_metrics["top1"] > best_val_top1:
        best_val_top1 = val_metrics["top1"]
        best_epoch = epoch
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": efficientnet_model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "val_loss": val_metrics["loss"],
                "val_top1": val_metrics["top1"],
                "val_top5": val_metrics["top5"],
                "class_names": class_names,
                "image_size": IMAGE_SIZE,
            },
            EFFICIENTNET_BEST_PATH,
        )
        checkpoint_note = " | saved new best checkpoint"
    else:
        checkpoint_note = ""

    with EFFICIENTNET_HISTORY_PATH.open("w", encoding="utf-8") as file:
        json.dump(efficientnet_history, file, indent=2)

    print(
        f"train loss={train_metrics['loss']:.4f}, "
        f"top1={train_metrics['top1']:.2f}%, top5={train_metrics['top5']:.2f}% | "
        f"val loss={val_metrics['loss']:.4f}, "
        f"top1={val_metrics['top1']:.2f}%, top5={val_metrics['top5']:.2f}%"
        f"{checkpoint_note}"
    )

best_checkpoint = torch.load(EFFICIENTNET_BEST_PATH, map_location=DEVICE)
if best_checkpoint["class_names"] != class_names:
    raise RuntimeError("EfficientNet checkpoint class order does not match class_names.json.")
efficientnet_model.load_state_dict(best_checkpoint["model_state_dict"])
efficientnet_model.eval()

print(f"\nBest EfficientNet epoch: {best_epoch}")
print(f"Best validation Top-1: {best_val_top1:.2f}%")
print(f"EfficientNet best checkpoint: {EFFICIENTNET_BEST_PATH}")
print(f"EfficientNet training history: {EFFICIENTNET_HISTORY_PATH}")
print("Step 2 complete: both models have been trained and their best checkpoints saved.")


# Step 3: Evaluation, comparative benchmarking, diagnostics, and export

This step reloads both **best** checkpoints and evaluates them on the same held-out samples from `test_loader`. Top-1 and Top-5 accuracy are computed over the complete test split. Latency is measured independently as preprocessed, batch-size-1 forward-pass time, with warm-up iterations and CUDA synchronization when a GPU is active. File size refers to the requested `.pt`/`.pth` checkpoint artifact.

Because `test_loader` contains ImageNet-normalized tensors for EfficientNet, its normalization is inverted before the tensors are passed to YOLO, which expects floating-point images in the `[0, 1]` range. This keeps the resized image content and test sample ordering identical for both models.


In [ ]:
import time

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.metrics import confusion_matrix

BENCHMARK_RESULTS_PATH = Path.cwd() / "benchmark_results.csv"
PLOTS_DIR = Path.cwd() / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

required_step3_inputs = (YOLO_BEST_PATH, EFFICIENTNET_BEST_PATH, CLASS_NAMES_PATH)
missing_step3_inputs = [str(path) for path in required_step3_inputs if not path.is_file()]
if missing_step3_inputs:
    raise FileNotFoundError(
        "Step 3 requires completed Step 1 and Step 2 artifacts. Missing: "
        + ", ".join(missing_step3_inputs)
    )

with CLASS_NAMES_PATH.open("r", encoding="utf-8") as file:
    saved_class_names = json.load(file)
if saved_class_names != class_names or len(saved_class_names) != EXPECTED_NUM_CLASSES:
    raise RuntimeError("class_names.json is inconsistent with the active test dataset.")

print(f"Benchmark device: {DEVICE}")
print(f"Test images: {len(test_dataset):,}")


## 1. Held-out test set evaluation

Both checkpoints are freshly loaded to ensure the benchmark evaluates the persisted best model rather than the final in-memory epoch. The loop also retains Top-1 predictions for the comparative confusion matrices.


In [ ]:
def load_trusted_torch_checkpoint(path, map_location):
    # weights_only was added after older Colab PyTorch releases; this file was created by Step 2.
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def first_tensor(model_output):
    if torch.is_tensor(model_output):
        return model_output
    if isinstance(model_output, (tuple, list)):
        for item in model_output:
            try:
                return first_tensor(item)
            except TypeError:
                continue
    if isinstance(model_output, dict):
        for item in model_output.values():
            try:
                return first_tensor(item)
            except TypeError:
                continue
    raise TypeError(f"Could not find a tensor in model output of type {type(model_output)}")


yolo_eval_wrapper = YOLO(str(YOLO_BEST_PATH))
yolo_eval_model = yolo_eval_wrapper.model.to(DEVICE).eval()

efficientnet_eval_model = efficientnet_b0(weights=None)
efficientnet_eval_model.classifier[1] = nn.Linear(
    efficientnet_eval_model.classifier[1].in_features, EXPECTED_NUM_CLASSES
)
efficientnet_checkpoint = load_trusted_torch_checkpoint(EFFICIENTNET_BEST_PATH, DEVICE)
if efficientnet_checkpoint["class_names"] != class_names:
    raise RuntimeError("EfficientNet checkpoint class order does not match class_names.json.")
efficientnet_eval_model.load_state_dict(efficientnet_checkpoint["model_state_dict"])
efficientnet_eval_model = efficientnet_eval_model.to(DEVICE).eval()

normalization_mean = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1, 3, 1, 1)
normalization_std = torch.tensor(IMAGENET_STD, device=DEVICE).view(1, 3, 1, 1)


def imagenet_to_unit_range(normalized_images):
    return (normalized_images * normalization_std + normalization_mean).clamp_(0.0, 1.0)


@torch.inference_mode()
def evaluate_both_models(yolo_model, efficientnet_model, data_loader, device):
    totals = {
        "yolo_top1": 0,
        "yolo_top5": 0,
        "efficientnet_top1": 0,
        "efficientnet_top5": 0,
        "samples": 0,
    }
    all_targets = []
    yolo_predictions = []
    efficientnet_predictions = []

    for normalized_images, targets in tqdm(data_loader, desc="test evaluation"):
        normalized_images = normalized_images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        efficientnet_scores = efficientnet_model(normalized_images)
        yolo_scores = first_tensor(yolo_model(imagenet_to_unit_range(normalized_images)))

        if efficientnet_scores.shape[1] != EXPECTED_NUM_CLASSES:
            raise RuntimeError(f"Unexpected EfficientNet output shape: {efficientnet_scores.shape}")
        if yolo_scores.shape[1] != EXPECTED_NUM_CLASSES:
            raise RuntimeError(f"Unexpected YOLO output shape: {yolo_scores.shape}")

        efficientnet_top5 = efficientnet_scores.topk(5, dim=1).indices
        yolo_top5 = yolo_scores.topk(5, dim=1).indices

        totals["efficientnet_top1"] += efficientnet_top5[:, 0].eq(targets).sum().item()
        totals["efficientnet_top5"] += efficientnet_top5.eq(targets.unsqueeze(1)).any(1).sum().item()
        totals["yolo_top1"] += yolo_top5[:, 0].eq(targets).sum().item()
        totals["yolo_top5"] += yolo_top5.eq(targets.unsqueeze(1)).any(1).sum().item()
        totals["samples"] += targets.size(0)

        all_targets.extend(targets.cpu().tolist())
        efficientnet_predictions.extend(efficientnet_top5[:, 0].cpu().tolist())
        yolo_predictions.extend(yolo_top5[:, 0].cpu().tolist())

    if totals["samples"] != len(test_dataset):
        raise RuntimeError(
            f"Evaluated {totals['samples']} samples, expected {len(test_dataset)}."
        )

    metrics = {
        "YOLOv8n-cls": {
            "top1": 100.0 * totals["yolo_top1"] / totals["samples"],
            "top5": 100.0 * totals["yolo_top5"] / totals["samples"],
        },
        "EfficientNet-B0": {
            "top1": 100.0 * totals["efficientnet_top1"] / totals["samples"],
            "top5": 100.0 * totals["efficientnet_top5"] / totals["samples"],
        },
    }
    return metrics, all_targets, yolo_predictions, efficientnet_predictions


test_metrics, test_targets, yolo_test_predictions, efficientnet_test_predictions = evaluate_both_models(
    yolo_eval_model, efficientnet_eval_model, test_loader, DEVICE
)
print("Held-out test evaluation complete.")


## 2. Latency and checkpoint-size benchmark

Latency excludes disk I/O and data-loader time. It measures one preprocessed 224×224 image per forward pass, includes framework dispatch, and synchronizes CUDA around the timed region. This is a reproducible relative benchmark for the active Colab runtime, not a substitute for profiling the eventual mobile ONNX runtime.


In [ ]:
LATENCY_WARMUP_RUNS = 10
LATENCY_TIMED_RUNS = 100


def synchronize_if_cuda(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


@torch.inference_mode()
def benchmark_forward_latency(forward_call, device, warmup_runs=10, timed_runs=100):
    for _ in range(warmup_runs):
        forward_call()
    synchronize_if_cuda(device)

    start_time = time.perf_counter()
    for _ in range(timed_runs):
        forward_call()
    synchronize_if_cuda(device)

    return (time.perf_counter() - start_time) * 1000.0 / timed_runs


latency_image, _ = test_dataset[0]
efficientnet_latency_input = latency_image.unsqueeze(0).to(DEVICE)
yolo_latency_input = imagenet_to_unit_range(efficientnet_latency_input.clone())

yolo_latency_ms = benchmark_forward_latency(
    lambda: first_tensor(yolo_eval_model(yolo_latency_input)),
    DEVICE,
    LATENCY_WARMUP_RUNS,
    LATENCY_TIMED_RUNS,
)
efficientnet_latency_ms = benchmark_forward_latency(
    lambda: efficientnet_eval_model(efficientnet_latency_input),
    DEVICE,
    LATENCY_WARMUP_RUNS,
    LATENCY_TIMED_RUNS,
)

def file_size_mb(path):
    return path.stat().st_size / (1024 ** 2)


benchmark_df = pd.DataFrame(
    [
        {
            "Model": "YOLOv8n-cls",
            "Top-1 Accuracy (%)": test_metrics["YOLOv8n-cls"]["top1"],
            "Top-5 Accuracy (%)": test_metrics["YOLOv8n-cls"]["top5"],
            "Average Latency (ms/image)": yolo_latency_ms,
            "Model File Size (MB)": file_size_mb(YOLO_BEST_PATH),
        },
        {
            "Model": "EfficientNet-B0",
            "Top-1 Accuracy (%)": test_metrics["EfficientNet-B0"]["top1"],
            "Top-5 Accuracy (%)": test_metrics["EfficientNet-B0"]["top5"],
            "Average Latency (ms/image)": efficientnet_latency_ms,
            "Model File Size (MB)": file_size_mb(EFFICIENTNET_BEST_PATH),
        },
    ]
).round(4)

benchmark_df.to_csv(BENCHMARK_RESULTS_PATH, index=False)
display(Markdown(benchmark_df.to_markdown(index=False)))
print(f"Saved benchmark metrics to: {BENCHMARK_RESULTS_PATH}")


## 3. Training curves and normalized confusion matrices

The curve figure compares all metrics available from the native training logs. Ultralytics records training/validation loss and validation Top-1 accuracy; EfficientNet records training and validation values for both metrics.

For readable confusion diagnostics, the 15 test classes with the greatest support are shown individually and every remaining class is grouped into `Other`. Predictions outside the displayed classes therefore remain visible instead of being discarded. Rows are normalized by the true-class count.


In [ ]:
if not EFFICIENTNET_HISTORY_PATH.is_file():
    raise FileNotFoundError(f"EfficientNet history not found: {EFFICIENTNET_HISTORY_PATH}")
with EFFICIENTNET_HISTORY_PATH.open("r", encoding="utf-8") as file:
    efficientnet_plot_history = json.load(file)

yolo_results_path = YOLO_RUN_DIR / "results.csv"
if not yolo_results_path.is_file():
    yolo_result_candidates = list((ARTIFACT_ROOT / "yolo").rglob("results.csv"))
    if not yolo_result_candidates:
        raise FileNotFoundError("Could not find the Ultralytics results.csv training log.")
    yolo_results_path = max(yolo_result_candidates, key=lambda path: path.stat().st_mtime)

yolo_history = pd.read_csv(yolo_results_path)
yolo_history.columns = yolo_history.columns.str.strip()
yolo_epochs = pd.to_numeric(yolo_history["epoch"], errors="raise").to_numpy()
if len(yolo_epochs) and yolo_epochs.min() == 0:
    yolo_epochs = yolo_epochs + 1
efficientnet_epochs = np.arange(1, len(efficientnet_plot_history["train_loss"]) + 1)


def yolo_metric(column_name, as_percentage=False):
    if column_name not in yolo_history.columns:
        return None
    values = pd.to_numeric(yolo_history[column_name], errors="coerce").to_numpy()
    if as_percentage and np.nanmax(np.abs(values)) <= 1.0001:
        values = values * 100.0
    return values


sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

axes[0].plot(efficientnet_epochs, efficientnet_plot_history["train_loss"], label="EfficientNet train")
axes[0].plot(efficientnet_epochs, efficientnet_plot_history["val_loss"], label="EfficientNet val")
yolo_train_loss = yolo_metric("train/loss")
yolo_val_loss = yolo_metric("val/loss")
if yolo_train_loss is not None:
    axes[0].plot(yolo_epochs, yolo_train_loss, label="YOLO train")
if yolo_val_loss is not None:
    axes[0].plot(yolo_epochs, yolo_val_loss, label="YOLO val")
axes[0].set(title="Training and validation loss", xlabel="Epoch", ylabel="Loss")
axes[0].legend()

axes[1].plot(efficientnet_epochs, efficientnet_plot_history["train_top1"], label="EfficientNet train")
axes[1].plot(efficientnet_epochs, efficientnet_plot_history["val_top1"], label="EfficientNet val")
yolo_val_top1 = yolo_metric("metrics/accuracy_top1", as_percentage=True)
if yolo_val_top1 is not None:
    axes[1].plot(yolo_epochs, yolo_val_top1, label="YOLO val")
axes[1].set(title="Top-1 accuracy", xlabel="Epoch", ylabel="Accuracy (%)")
axes[1].set_ylim(0, 100)
axes[1].legend()

fig.suptitle("GuardPlant training diagnostics", fontsize=15)
fig.tight_layout()
TRAINING_CURVES_PATH = PLOTS_DIR / "training_curves.png"
fig.savefig(TRAINING_CURVES_PATH, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved training curves to: {TRAINING_CURVES_PATH}")


In [ ]:
TOP_CONFUSION_CLASSES = 15
target_array = np.asarray(test_targets, dtype=np.int64)
class_support = np.bincount(target_array, minlength=EXPECTED_NUM_CLASSES)
top_class_ids = np.argsort(-class_support, kind="stable")[:TOP_CONFUSION_CLASSES]
top_class_to_plot_index = {int(class_id): index for index, class_id in enumerate(top_class_ids)}
other_index = TOP_CONFUSION_CLASSES
confusion_labels = [class_names[int(class_id)] for class_id in top_class_ids] + ["Other"]


def map_to_top_classes(values):
    return np.asarray(
        [top_class_to_plot_index.get(int(value), other_index) for value in values],
        dtype=np.int64,
    )


mapped_targets = map_to_top_classes(test_targets)
mapped_yolo_predictions = map_to_top_classes(yolo_test_predictions)
mapped_efficientnet_predictions = map_to_top_classes(efficientnet_test_predictions)
matrix_label_ids = np.arange(TOP_CONFUSION_CLASSES + 1)

yolo_confusion = confusion_matrix(
    mapped_targets, mapped_yolo_predictions, labels=matrix_label_ids, normalize="true"
)
efficientnet_confusion = confusion_matrix(
    mapped_targets, mapped_efficientnet_predictions, labels=matrix_label_ids, normalize="true"
)

fig, axes = plt.subplots(1, 2, figsize=(25, 10), constrained_layout=True)
for axis, matrix, title in (
    (axes[0], yolo_confusion, "YOLOv8n-cls"),
    (axes[1], efficientnet_confusion, "EfficientNet-B0"),
):
    sns.heatmap(
        matrix,
        ax=axis,
        cmap="Blues",
        vmin=0.0,
        vmax=1.0,
        xticklabels=confusion_labels,
        yticklabels=confusion_labels,
        cbar_kws={"label": "Fraction of true class"},
    )
    axis.set_title(f"{title} — normalized confusion matrix")
    axis.set_xlabel("Predicted class")
    axis.set_ylabel("True class")
    axis.tick_params(axis="x", labelrotation=60)
    axis.tick_params(axis="y", labelrotation=0)

CONFUSION_MATRIX_PATH = PLOTS_DIR / "confusion_matrix.png"
fig.savefig(CONFUSION_MATRIX_PATH, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved normalized confusion matrices to: {CONFUSION_MATRIX_PATH}")


## 4. Export production models to ONNX

Both exports use an input size of `[1, 3, 224, 224]`, ONNX opset 17, and named input/output tensors. EfficientNet permits a dynamic batch dimension. Ultralytics embeds its own model metadata in the YOLO ONNX artifact. Each exported graph is checked with the ONNX validator before backup.


In [ ]:
import onnx

YOLO_ONNX_PATH = Path.cwd() / "yolov8n_guardplant.onnx"
EFFICIENTNET_ONNX_PATH = Path.cwd() / "efficientnet_b0_guardplant.onnx"

yolo_export_wrapper = YOLO(str(YOLO_BEST_PATH))
ultralytics_onnx_path = Path(
    yolo_export_wrapper.export(
        format="onnx",
        imgsz=IMAGE_SIZE,
        batch=1,
        dynamic=False,
        simplify=False,
        opset=17,
        device="cpu",
    )
)
if not ultralytics_onnx_path.is_file():
    raise FileNotFoundError(f"Ultralytics ONNX export was not created: {ultralytics_onnx_path}")
if ultralytics_onnx_path.resolve() != YOLO_ONNX_PATH.resolve():
    shutil.copy2(ultralytics_onnx_path, YOLO_ONNX_PATH)

efficientnet_export_model = efficientnet_eval_model.to("cpu").eval()
dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, dtype=torch.float32)
torch.onnx.export(
    efficientnet_export_model,
    dummy_input,
    EFFICIENTNET_ONNX_PATH,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["images"],
    output_names=["logits"],
    dynamic_axes={"images": {0: "batch"}, "logits": {0: "batch"}},
    dynamo=False,
)

for exported_model_path in (YOLO_ONNX_PATH, EFFICIENTNET_ONNX_PATH):
    onnx_model = onnx.load(str(exported_model_path))
    onnx.checker.check_model(onnx_model)
    print(f"Validated ONNX: {exported_model_path} ({file_size_mb(exported_model_path):.2f} MB)")


## 5. Back up production artifacts to Google Drive

The final ONNX models, canonical class mapping, and benchmark table are copied to `/content/drive/MyDrive/GuardPlant_Artifacts/`. Google Drive was mounted in Step 1.


In [ ]:
DRIVE_ARTIFACT_DIR = Path("/content/drive/MyDrive/GuardPlant_Artifacts")
DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

artifacts_to_back_up = (
    YOLO_ONNX_PATH,
    EFFICIENTNET_ONNX_PATH,
    CLASS_NAMES_PATH,
    BENCHMARK_RESULTS_PATH,
)

for source_path in artifacts_to_back_up:
    if not source_path.is_file():
        raise FileNotFoundError(f"Cannot back up missing artifact: {source_path}")
    destination_path = DRIVE_ARTIFACT_DIR / source_path.name
    shutil.copy2(source_path, destination_path)
    print(f"Backed up: {destination_path}")

print("Step 3 complete: evaluation, diagnostics, ONNX export, and Drive backup finished.")
